In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr
from sklearn.model_selection import RandomizedSearchCV

In [2]:
df_train = pd.read_csv("data/train_final.csv")

In [3]:
df_train.head()

,cust_id,sale_id_nunique,prod_id_count,sale_revenue_sum,sale_revenue_mean,sale_discount_applied_sum,is_returned_mean,is_returned_last,pack_delay_days_max,is_jan_july_mean,prod_web_only_mean,prod_brand_mode,prod_size_nunique,recency_days,discount_affinity,recent_spend_ratio,spend_trajectory_ratio,churn_risk_ratio,revenue_2018_2019
0,222agnowc53dykbq,1,1,89.95,89.950000,0.00,0.000000,0,0,0.000000,0.000000,geox,1,383,0.000000,0.0,0.000000,383.000000,0.00
1,222ny4m63rmalpdl,1,3,125.93,41.976667,-45.16,0.333333,0,0,0.000000,0.000000,other,3,374,-0.358612,0.0,0.000000,37400.000000,0.00
2,223xvc4rbjatlnev,1,2,116.14,58.070000,-49.76,0.000000,0,7,1.000000,0.000000,teva,2,520,-0.428448,0.0,0.000000,52000.000000,0.00
3,223y2r357elerbis,1,2,47.97,23.985000,-58.96,0.500000,0,0,1.000000,0.000000,tamaris,1,348,-1.229102,0.0,479700.000000,34800.000000,0.00
4,2245y4r7mgo45qg3,8,9,533.92,59.324444,-111.83,0.111111,0,5,0.222222,0.222222,other,5,169,-0.209451,0.0,0.909175,2.666667,645.49


In [4]:
X_train = df_train.drop(columns=["cust_id", "revenue_2018_2019"])

In [5]:
y_train = df_train["revenue_2018_2019"]

In [6]:
categorical_cols = ["prod_brand_mode"]

In [7]:
for col in categorical_cols:
    X_train[col] = X_train[col].astype("category")

## training the model

In [8]:
model = lgb.LGBMRegressor(random_state=15, n_estimators=300, objective="mae", n_jobs=-1)

In [9]:
param_grid = {
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "num_leaves": [31, 50, 75, 100],
    "max_depth": [5, 7, 10, -1],
    "min_child_samples": [20, 50, 100, 150],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],  # Percentage of features used per tree
    "subsample": [0.7, 0.8, 0.9, 1.0],  # Percentage of rows used per tree
}

In [10]:
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=20,
    scoring="neg_mean_absolute_error",
    cv=3,
    verbose=0,
    n_jobs=1,
    random_state=15,
)

In [11]:
print("Starting the hyperparameter tuning ...")
random_search.fit(X_train, y_train)

Starting the hyperparameter tuning ...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.036258 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2491
[LightGBM] [Info] Number of data points in the train set: 62181, number of used features: 17


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.028596 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2477
[LightGBM] [Info] Number of data points in the train set: 62181, number of used features: 17
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003379 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2492
[LightGBM] [Info] Number of data points in the train set: 62182, number of used features: 17
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001500 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2491
[LightGBM] [Info] Number of data p

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LGBMRegressor...ndom_state=15)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.7, 0.8, ...], 'learning_rate': [0.01, 0.03, ...], 'max_depth': [5, 7, ...], 'min_child_samples': [20, 50, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategie

In [12]:
print("\nBest Parameters Found:")
for key, value in random_search.best_params_.items():
    print(f"  {key}: {value}")


Best Parameters Found:
  subsample: 0.9
  num_leaves: 100
  min_child_samples: 150
  max_depth: 5
  learning_rate: 0.03
  colsample_bytree: 0.9


## validating the model

In [13]:
df_valid = pd.read_csv("data/valid_final.csv")

In [14]:
X_val = df_valid.drop(columns=["cust_id", "revenue_2018_2019"])
y_val = df_valid["revenue_2018_2019"]

In [15]:
for col in categorical_cols:
    X_val[col] = X_val[col].astype("category")

In [16]:
y_pred = random_search.best_estimator_.predict(X_val)

In [17]:
y_pred = np.clip(y_pred, a_min=0, a_max=None)

In [18]:
mae = mean_absolute_error(y_val, y_pred)

In [19]:
spearman_corr, p_value = spearmanr(y_val, y_pred)

In [20]:
print("-" * 30)
print(f"Validation MAE: %{mae:.2f}")
print(f"Validation Spearman Correlation: {spearman_corr:.4f}")
print("-" * 30)

------------------------------
Validation MAE: %63.24
Validation Spearman Correlation: 0.3966
------------------------------


In [21]:
importance_df = pd.DataFrame(
    {"Feature": X_train.columns, "Importance": random_search.best_estimator_.feature_importances_} # type: ignore
).sort_values(by="Importance", ascending=False)

print("\nFeature Importance Table:")
print(importance_df)


Feature Importance Table:
                      Feature  Importance
10            prod_brand_mode        1490
12               recency_days        1128
16           churn_risk_ratio        1056
2            sale_revenue_sum         782
1               prod_id_count         739
15     spend_trajectory_ratio         541
5            is_returned_mean         465
3           sale_revenue_mean         399
4   sale_discount_applied_sum         337
0             sale_id_nunique         282
13          discount_affinity         246
6            is_returned_last         110
7         pack_delay_days_max          90
11          prod_size_nunique          73
8            is_jan_july_mean          65
9          prod_web_only_mean          46
14         recent_spend_ratio          36


## getting the test results

In [22]:
df_test = pd.read_csv("data/test_final.csv")

In [23]:
test_ids = df_test["cust_id"]

In [24]:
X_test = df_test.drop(columns=["cust_id"])

In [26]:
if "prod_brand_mode" in X_test.columns:
    X_test["prod_brand_mode"] = X_test["prod_brand_mode"].astype('category')

In [27]:
preds = random_search.best_estimator_.predict(X_test)

In [28]:
preds = np.clip(preds, a_min=0, a_max=None)

In [29]:
submission = pd.DataFrame({
    "cust_id": test_ids,
    "revenue": preds
})

In [30]:
submission.to_csv("data/leaderboard_submission.csv", index=False)

In [31]:
submission.head()

,cust_id,revenue
0,222wlefm7esnsi3h,0.169280
1,224j2bblkxseyuux,2.981475
2,224myrd7nlqdkm7j,0.000000
3,224qcoczcvqlu56j,42.250419
4,224rajycdiuglkxy,0.048112
